# SegNet from scratch — training notebook (Kaggle & Colab)

Train SegNet on a **free Kaggle or Colab GPU**. The notebook installs the
[`seg_net_from_scratch`](https://github.com/benisalla/seg_net_from_scratch)
package and uses the same code as the command line `segnet-train`.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 / P100* | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* | on by default |
| Mapillary data | *Add Input* → search "Mapillary Vistas" | upload to Google Drive |

Choose the dataset in **2. Config**, then press **Run all**.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/benisalla/seg_net_from_scratch.git

## 2. Config

- `"shapes"`: synthetic toy data, no download. Learns in a few minutes and
  checks that everything works.
- `"pets"`: Oxford-IIIT Pet, pet vs background (~800 MB download).
- `"mapillary"`: Mapillary Vistas street scenes. `MAPILLARY_ROOT = None`
  searches `/kaggle/input` for the dataset.

In [ ]:
DATASET = "shapes"          # "shapes" | "pets" | "mapillary"
MAPILLARY_ROOT = None       # None = search /kaggle/input; or e.g. "/content/drive/MyDrive/mapillary"
CLASSES = None              # mapillary only, e.g. ["object--traffic-sign--front", "object--traffic-sign--back"]

PRESETS = {  # image_size, base_channels, epochs, batch_size, lr
    "shapes":    dict(image_size=64,  base_channels=16, epochs=20, batch_size=32, lr=3e-3),
    "pets":      dict(image_size=128, base_channels=32, epochs=30, batch_size=32, lr=1e-3),
    "mapillary": dict(image_size=256, base_channels=64, epochs=60, batch_size=16, lr=1e-3),
}
P = PRESETS[DATASET]
OUTPUT_DIR = f"/kaggle/working/checkpoints/{DATASET}" if __import__("os").path.isdir("/kaggle") else f"checkpoints/{DATASET}"
print(DATASET, P, OUTPUT_DIR)

## 3. Find the data

In [ ]:
import glob, os

def find_mapillary(start="/kaggle/input"):
    """Folder that holds training/ or validation/ with images/ inside."""
    for images in sorted(glob.glob(f"{start}/**/images", recursive=True)):
        split = os.path.dirname(images)
        if os.path.basename(split) in ("training", "validation"):
            return os.path.dirname(split)
    raise FileNotFoundError(f"no Mapillary training/ or validation/ folder under {start}")

if DATASET == "mapillary":
    ROOT = MAPILLARY_ROOT or find_mapillary()
elif DATASET == "pets":
    ROOT = "/kaggle/working/data" if os.path.isdir("/kaggle") else "data"
else:
    ROOT = "data"
print("data root:", ROOT)

## 4. Look at a few samples

In [ ]:
import matplotlib.pyplot as plt
from segnet.config import DataConfig
from segnet.data import build_datasets, denormalize
from segnet.viz import colorize

data_cfg = DataConfig(dataset=DATASET, root=ROOT, image_size=P["image_size"], classes=CLASSES, num_workers=2)
train_set, val_set, info = build_datasets(data_cfg)
print(f"train {len(train_set)}  val {len(val_set)}  classes {info.num_classes}")

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    image, mask = train_set[i]
    axes[0, i].imshow(denormalize(image).permute(1, 2, 0))
    axes[1, i].imshow(colorize(mask, info.colors))
    axes[0, i].axis("off"); axes[1, i].axis("off")
plt.show()

## 5. Sanity check: memorise 8 images

A model that cannot overfit 8 images is broken. The loss should drop close to
0 and pixel accuracy should approach 1. This takes about a minute on a GPU.

In [ ]:
from segnet.config import ModelConfig, TrainConfig
from segnet.train import train

check = train(data_cfg, ModelConfig(base_channels=16),
              TrainConfig(epochs=100, batch_size=8, lr=3e-3, overfit=8, log_every=0,
                          output_dir=OUTPUT_DIR + "_overfit"),
              datasets=(train_set, val_set, info))
print(f"pixel accuracy on the 8 memorised images: {check['pixel_acc']:.3f}")

## 6. Train

Checkpoints are saved every epoch. If the session stops, set
`RESUME = OUTPUT_DIR + "/last.pt"` and run this cell again.

In [ ]:
RESUME = None

model_cfg = ModelConfig(base_channels=P["base_channels"])
train_cfg = TrainConfig(epochs=P["epochs"], batch_size=P["batch_size"], lr=P["lr"],
                        dice_weight=0.5 if DATASET == "mapillary" else 0.0,
                        log_every=100, output_dir=OUTPUT_DIR)
result = train(data_cfg, model_cfg, train_cfg, datasets=(train_set, val_set, info), resume=RESUME)
print(f"best validation mIoU: {result['best_miou']:.3f}")

## 7. Results

In [ ]:
from IPython.display import Image as Show, display
display(Show(f"{OUTPUT_DIR}/history.png"))
display(Show(f"{OUTPUT_DIR}/predictions.png"))

### Per-class IoU on the validation set

In [ ]:
import torch
from torch.utils.data import DataLoader
from segnet.metrics import ConfusionMatrix
from segnet.train import load_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model, info, _ = load_model(f"{OUTPUT_DIR}/best.pt", device)
cm = ConfusionMatrix(info.num_classes)
with torch.no_grad():
    for images, masks in DataLoader(val_set, batch_size=32, num_workers=2):
        cm.update(model(images.to(device)).argmax(1).cpu(), masks)
print(cm.compute())
for name, iou in zip(info.class_names, cm.iou().tolist()):
    print(f"{name:45s} {iou:.3f}")  # nan = class not in the validation set

## 8. Download the trained model

In [ ]:
import shutil
archive = shutil.make_archive(f"segnet_{DATASET}", "zip", OUTPUT_DIR)
print("saved", archive, "(Kaggle: Output tab after 'Save Version'; Colab: Files panel)")